How ChromaDB Retrieves

1.   User asks a question
2.   Questions is converted to vector
3. ChromaDB compares query vector against all stored vectors
4. Returns top k most similar documents
5. Sorted by distance(lower=more similar)



In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully")
print("Ready to build a RAG system")

All libraries imported successfully
Ready to build a RAG system


In [ ]:
GROQ_API_KEY="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"
os.environ['GROQ_API_KEY']=GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq API client intialized")
print("Note: If you see an authentication error later, double-check your API key.")

Groq API client intialized
Note: If you see an authentication error later, double-check your API key.


In [ ]:
df=pd.read_csv("/content/college_notes.csv")
print("Shape of the dataset: ",df.shape)
print("\nColumn names: ",df.columns.tolist())


Shape of the dataset:  (15, 4)

Column names:  ['note_id', 'subject', 'topic', 'content']


In [ ]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content (number of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python

In [ ]:
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared: {len(documents)}")
print(f"First document ID : {ids[0]}")
print(f"First metadata : {metadatas[0]}")
print(f"First 100 chars of doc: {documents[0][:100]}...")

Total chunks prepared: 15
First document ID : note_N001
First metadata : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [ ]:
print("Loading embedding model...")
print("(This may take 30-60 seconds on first run-model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("\nEmbedding model loaded successfully")
test_embedding=embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of  test embedding: {test_embedding[:5]}")

Loading embedding model...
(This may take 30-60 seconds on first run-model is being downloaded)
(Subsequent runs will be faster as the model is cached)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded successfully
Test embedding shape: (384,)
First 5 values of  test embedding: [0.08429645 0.05795371 0.00449334 0.10582108 0.00708343]


In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDB client cerated.")
print(f"Collection name: {collection.name}")
print(f"Number of documents in collection: {collection.count()}")

ChromaDB client cerated.
Collection name: college_notes_rag
Number of documents in collection: 15


In [ ]:
print("Generating embeddings for all 15 notes..")
print("This may take 15-30 seconds")
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\nEmbedding matrix shape:{embeddings.shape}")
print(collection.count())

Generating embeddings for all 15 notes..
This may take 15-30 seconds


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape:(15, 384)
15


In [ ]:
print("Adding documents to ChromaDB collection...")
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(), # Convert numpy array to list
    metadatas=metadatas,
    ids=ids
)
print(f"Number of documents in collection after adding: {collection.count()}")

Adding documents to ChromaDB collection...
Number of documents in collection after adding: 15


In [ ]:
def retrieve_relavent_chunks(question,top_k=3):
  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=question_embedding,
      n_results=top_k,
  )
  return results
print("Retrieval function defined successfully.")
print("Function: retrieve_relavent_chunks(question,top_k=3)")

Retrieval function defined successfully.
Function: retrieve_relavent_chunks(question,top_k=3)


In [ ]:
test_question="What is ETL and how does it work in data engineering?"
print(f"Test Quetion: {test_question}")
print("="*60)
results=retrieve_relavent_chunks(test_question,top_k=3)
print("\nTop 3 relavent chunks")
print("="*60)
for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"\nResult {i+1}:")
  print(f"  Subject  : {meta['subject']}")
  print(f"  Topic    : {meta['topic']}")
  print(f"  Distance : {dist}")
  print(f"  Content  : {doc[:120]}...")

Test Quetion: What is ETL and how does it work in data engineering?

Top 3 relavent chunks

Result 1:
  Subject  : Data Engineering
  Topic    : ETL Pipelines
  Distance : 0.22689150273799896
  Content  : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2:
  Subject  : Data Engineering
  Topic    : APIs and Data Collection
  Distance : 1.0689688920974731
  Content  : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3:
  Subject  : Python Programming
  Topic    : Data Visualization
  Distance : 1.3374853134155273
  Content  : Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


In [ ]:
def build_context_from_results(results):
  context_parts=[]
  for i, (doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    chunk_text=f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  context_str="\n\n--\n\n".join(context_parts)
  return context_str
context=build_context_from_results(results)
print("Built context stirng from retrieved chunks:")
print("="*60)
print(context[:500]+"...")
print(f"\nTotal context length : {len(context)} characters")

Built context stirng from retrieved chunks:
[Source 1: Data Engineering - ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

--

[Source 2: Data Engineering - APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like w...

Total context length : 854 characters


In [ ]:
def generate_rag_answer(question,context):
  system_prompt="""You are helpful academic assistant for engineering students.
  You will be given context retrieved from a college knowledge base, and a student's question.

  RULES:
  1.Answer only using the information provided in the context below.
  2.If the answer is not found in the context, say exactly:
  "I dont have enough information in my knowledge base to answer this question."
  3.Do not use your general training knowledge.
  4.Keep answers clear, accurate, and beginner-friendly.
  5.Mention which source the information came from when possible."""
  user_prompt=f"""Context from knowledge base:
{context}

---
Students Question: {question}

Pleasse answer the question based only on the context provided above."""
  response=groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
  )
  answer=response.choices[0].message.content
  return answer
print("RAG generation function defined.")

RAG generation function defined.


In [ ]:
def ask_college_assistant(question,top_k=3,verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("="*60)
    print("Step 1: Retriving relavant documents...")
  results=retrieve_relavent_chunks(question,top_k=top_k)

  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i,meta in enumerate(results['metadatas'][0]):
      print(f" {i+1}. {meta['subject']}-{meta['topic']}")
    print("\nStep 2: building context string...")
  context=build_context_from_results(results)
  if verbose:
    print(f"Context built ({len(context)} characters)")
    print("\nStep 3: Sending to LLM for answer generation...")
  answer=generate_rag_answer(question,context)
  if verbose:
    print("\n"+"="*60)
    print("ANSWER:")
    print("="*60)
    print(answer)
    print("="*60)
  return answer
print("Complete RAG pipeline function ready")
print("Function: ask_college_assistant(question,top_k=3)")

Complete RAG pipeline function ready
Function: ask_college_assistant(question,top_k=3)


In [ ]:
question_1="What is ETL and what are its three main stages?"
answer_1=ask_college_assistant(question_1,top_k=3,verbose=True)

Question: What is ETL and what are its three main stages?
Step 1: Retriving relavant documents...
Retrieved 3 chunks from the knowledge base:
 1. Data Engineering-ETL Pipelines
 2. Generative AI-Retrieval Augmented Generation
 3. Generative AI-Prompt Engineering

Step 2: building context string...
Context built (928 characters)

Step 3: Sending to LLM for answer generation...

ANSWER:
Based on the context provided, the answer to the student's question is:

ETL stands for Extract Transform Load. The three main stages of ETL are:

1. Extract: This stage involves collecting raw data from different sources.
2. Transform: This stage involves transforming the raw data into a clean and structured format.
3. Load: This stage involves loading the transformed data into a database or data warehouse for analysis.

This information comes from [Source 1: Data Engineering - ETL Pipelines].


In [ ]:
question_3="What is the population of Tokyo?"
print("Testing with an out-of-scope question (not in college notes):")
answer_3=ask_college_assistant(question_3,top_k=3,verbose=True)

Testing with an out-of-scope question (not in college notes):
Question: What is the population of Tokyo?
Step 1: Retriving relavant documents...
Retrieved 3 chunks from the knowledge base:
 1. Generative AI-Large Language Models
 2. Data Engineering-SQL Databases
 3. Data Engineering-Data Cleaning

Step 2: building context string...
Context built (797 characters)

Step 3: Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.


In [ ]:
def retrieve_by_subject(subject,top_k=3):
  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
      where={"subject":subject_filter}
  )
  return results
  print("Retrieving only from GenAI subject:")
  print("")



1.   What is hallucination in the context of LLMs?
2.   What does RAG stand for? What problem does it solve?
3. What is the role of a vector database in the RAG pipeline?





1. What is the difference between the Indexing phase and the Querying phase of RAG?
2. Why must you use the same embedding model for both documents and queries?
3. Why is a low temperature (e.g. 0.1) preferred for RAG-based LLM calls?





1. Modify the ask_college_assistant() function to also display the distance scores of retrieved chunks in the output.

2. Change the system prompt in generate_rag_answer() to instruct the LLM to always respond in bullet points.

3. Add a function that returns only the topic names of retrieved chunks without their full content.